# Building a GPT from First Principles
### Trained on Little Shakespeare Dataset

This notebook implements a GPT-style (decoder-only) Transformer from scratch.  
It covers:
- Character-level tokenization
- Token + Positional Embeddings
- Multi-Head Causal Self-Attention
- Feed-Forward Network (FFN)
- Residual connections + Layer Normalization
- Decoder-only Transformer stack
- AdamW optimizer with warmup + cosine LR schedule
- Gradient clipping
- Train/Val loss tracking
- Text generation via autoregressive sampling


## 1. Install & Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import urllib.request
import matplotlib.pyplot as plt

# Check GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

## 2. Download & Explore the Dataset
We use the **tinyshakespeare** dataset (~1 MB of Shakespeare text).

In [ ]:
# Download tinyshakespeare
url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
urllib.request.urlretrieve(url, 'input.txt')

with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print(f'Total characters: {len(text):,}')
print(f'Sample text:\n{text[:300]}')

## 3. Character-Level Tokenization
We use **character-level** tokenization: vocabulary = all unique characters in the text (65 chars).
- No out-of-vocabulary tokens
- Model learns spelling, punctuation, and formatting

In [ ]:
# Build vocabulary
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(f'Vocabulary size: {vocab_size}')
print(f'Characters: {repr("".join(chars))}')

# Character <-> integer mappings
stoi = {ch: i for i, ch in enumerate(chars)}  # string to int
itos = {i: ch for i, ch in enumerate(chars)}  # int to string

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

# Encode full text to tensor
data = torch.tensor(encode(text), dtype=torch.long)
print(f'Data tensor shape: {data.shape}')

## 4. Train / Validation Split
- **90%** training, **10%** validation
- Validation = held-out text → detects overfitting early

In [ ]:
split = int(0.9 * len(data))
train_data = data[:split]
val_data   = data[split:]

print(f'Train tokens: {len(train_data):,}')
print(f'Val tokens:   {len(val_data):,}')

## 5. Hyperparameters

In [ ]:
# ── Model ──────────────────────────────────────────
block_size   = 128    # context length (chars per chunk)
d_model      = 256    # embedding dimension
n_heads      = 8      # number of attention heads
n_layers     = 6      # number of decoder blocks
d_ff         = 4 * d_model  # FFN hidden dim (standard: 4x d_model)
dropout_rate = 0.1

# ── Training ───────────────────────────────────────
batch_size   = 64
max_steps    = 5000
eval_interval= 500    # evaluate every N steps
eval_iters   = 100    # batches to average for loss estimate
checkpoint_interval = 500

# ── Optimizer ──────────────────────────────────────
learning_rate  = 3e-4
weight_decay   = 0.1
grad_clip      = 1.0  # gradient clipping threshold

# ── LR Schedule ────────────────────────────────────
warmup_steps   = 200  # LR linearly increases for this many steps
# After warmup: cosine decay to 10% of peak LR

print('Hyperparameters set.')
print(f'  block_size={block_size}, d_model={d_model}, n_heads={n_heads}, n_layers={n_layers}')

## 6. Data Loader
Creates random batches of (input, target) chunks.  
Input = chars 0–127, Target = chars 1–128 (shifted by 1 — next-token prediction).

In [ ]:
def get_batch(split):
    """
    Sample a random batch of (input, target) sequences.
    Target is input shifted by 1 position — standard next-token prediction.
    """
    source = train_data if split == 'train' else val_data
    # Random starting indices
    ix = torch.randint(len(source) - block_size, (batch_size,))
    x  = torch.stack([source[i : i + block_size]     for i in ix])
    y  = torch.stack([source[i + 1 : i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

# Sanity check
xb, yb = get_batch('train')
print(f'Input batch shape:  {xb.shape}  (batch_size x block_size)')
print(f'Target batch shape: {yb.shape}')

## 7. Model Architecture

### 7.1 — Multi-Head Causal Self-Attention
Each token plays three roles:
- **Query (Q)**: what am I looking for?
- **Key (K)**: what do I represent?
- **Value (V)**: here is my content

Causal masking ensures token `i` can only attend to positions `≤ i` (no future leakage).

In [ ]:
class MultiHeadCausalAttention(nn.Module):
    """
    Multi-head scaled dot-product self-attention with causal (autoregressive) mask.

    For each head:
        score(i,j) = (q_i · k_j) / sqrt(d_k)
        a(i,j)     = softmax(score)   [future positions masked to -inf]
        output_i   = sum_j  a(i,j) * v_j

    Heads are computed in parallel by reshaping the batch dimension.
    """

    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0, 'd_model must be divisible by n_heads'

        self.n_heads = n_heads
        self.d_k     = d_model // n_heads  # dimension per head

        # Single projection for Q, K, V (3x for efficiency)
        self.qkv_proj = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.attn_drop = nn.Dropout(dropout)
        self.resid_drop = nn.Dropout(dropout)

        # Causal mask: upper triangle = -inf so future tokens are invisible
        self.register_buffer(
            'mask',
            torch.triu(torch.ones(block_size, block_size), diagonal=1).bool()
        )

    def forward(self, x):
        B, T, C = x.shape  # (batch, time/seq_len, channels/d_model)

        # Project to Q, K, V
        qkv = self.qkv_proj(x)               # (B, T, 3*C)
        q, k, v = qkv.split(C, dim=-1)        # each (B, T, C)

        # Reshape for multi-head: (B, n_heads, T, d_k)
        def split_heads(t):
            return t.view(B, T, self.n_heads, self.d_k).transpose(1, 2)

        q, k, v = split_heads(q), split_heads(k), split_heads(v)

        # Scaled dot-product attention
        # score(i,j) = q_i · k_j / sqrt(d_k)
        scale  = math.sqrt(self.d_k)
        scores = torch.matmul(q, k.transpose(-2, -1)) / scale  # (B, H, T, T)

        # Apply causal mask: set future positions to -inf
        scores = scores.masked_fill(self.mask[:T, :T], float('-inf'))

        # Softmax → attention weights (sum to 1 per row)
        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.attn_drop(attn_weights)

        # Weighted sum of values
        out = torch.matmul(attn_weights, v)           # (B, H, T, d_k)

        # Merge heads back: (B, T, C)
        out = out.transpose(1, 2).contiguous().view(B, T, C)

        return self.resid_drop(self.out_proj(out))

### 7.2 — Feed-Forward Network (FFN)
Applied identically to each token position after attention.  
`FFN(x) = W2 · GELU(W1·x + b1) + b2`

In [ ]:
class FeedForward(nn.Module):
    """
    Position-wise Feed-Forward Network.
    Two linear layers with GELU activation in between.
    Hidden dimension is typically 4 × d_model.
    """

    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

### 7.3 — Decoder Block
One block = Self-Attention → Add & Norm → FFN → Add & Norm  
Residual connections (`x + sublayer(x)`) stabilize gradient flow.

In [ ]:
class DecoderBlock(nn.Module):
    """
    Single GPT decoder block:
        x = x + Attention(LayerNorm(x))    # residual + self-attention
        x = x + FFN(LayerNorm(x))          # residual + feed-forward

    LayerNorm is applied BEFORE each sublayer (Pre-LN) for training stability.
    """

    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.ln1  = nn.LayerNorm(d_model)
        self.attn = MultiHeadCausalAttention(d_model, n_heads, dropout)
        self.ln2  = nn.LayerNorm(d_model)
        self.ffn  = FeedForward(d_model, d_ff, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))   # Self-attention with residual
        x = x + self.ffn(self.ln2(x))    # FFN with residual
        return x

### 7.4 — Full GPT Model
Stacks N decoder blocks on top of token + positional embeddings.

In [ ]:
class GPT(nn.Module):
    """
    Decoder-only GPT model.

    Architecture:
        Input tokens
          → Token Embedding  (vocab_size → d_model)
          + Position Embedding (block_size → d_model)
          → Dropout
          → N × DecoderBlock
          → LayerNorm
          → Linear head  (d_model → vocab_size)
          → Logits (used with cross-entropy loss)
    """

    def __init__(self, vocab_size, block_size, d_model, n_heads,
                 n_layers, d_ff, dropout=0.1):
        super().__init__()
        self.block_size = block_size

        # Embeddings
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(block_size, d_model)
        self.emb_drop = nn.Dropout(dropout)

        # Decoder stack
        self.blocks = nn.ModuleList([
            DecoderBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])

        # Final layer norm + language model head
        self.ln_f  = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

        # Weight tying: token embedding and lm_head share weights (common in GPT)
        self.lm_head.weight = self.tok_emb.weight

        # Parameter initialization
        self.apply(self._init_weights)

        total_params = sum(p.numel() for p in self.parameters())
        print(f'GPT initialized | Layers: {n_layers} | Heads: {n_heads} | '
              f'd_model: {d_model} | Total params: {total_params:,}')

    def _init_weights(self, module):
        """Initialize weights following GPT-2 conventions."""
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.LayerNorm):
            nn.init.ones_(module.weight)
            nn.init.zeros_(module.bias)

    def forward(self, idx, targets=None):
        """
        Forward pass.
        idx:     (B, T) token indices
        targets: (B, T) next-token targets (optional, for loss computation)
        """
        B, T = idx.shape
        assert T <= self.block_size, f'Sequence length {T} exceeds block_size {self.block_size}'

        # Token + Positional embeddings
        positions = torch.arange(T, device=idx.device)   # (T,)
        x = self.emb_drop(self.tok_emb(idx) + self.pos_emb(positions))  # (B, T, d_model)

        # Pass through all decoder blocks
        for block in self.blocks:
            x = block(x)

        # Final layer norm
        x = self.ln_f(x)                    # (B, T, d_model)

        # Language model head → logits over vocabulary
        logits = self.lm_head(x)            # (B, T, vocab_size)

        # Compute cross-entropy loss if targets provided
        loss = None
        if targets is not None:
            # Flatten for F.cross_entropy: (B*T, vocab_size) vs (B*T,)
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1)
            )

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        """
        Autoregressive text generation.
        Starts from a context `idx` and generates `max_new_tokens` new characters.
        - temperature: >1 = more random, <1 = more deterministic
        - top_k: restrict sampling to top-k most likely tokens
        """
        self.eval()
        for _ in range(max_new_tokens):
            # Crop context to block_size
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            # Focus on last time step
            logits = logits[:, -1, :] / temperature  # (B, vocab_size)

            # Optional top-k filtering
            if top_k is not None:
                topk_vals, _ = torch.topk(logits, top_k)
                threshold = topk_vals[:, -1].unsqueeze(-1)
                logits = logits.masked_fill(logits < threshold, float('-inf'))

            # Sample from distribution
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)  # (B, 1)
            idx = torch.cat([idx, next_token], dim=1)              # (B, T+1)

        return idx

## 8. Instantiate Model

In [ ]:
model = GPT(
    vocab_size  = vocab_size,
    block_size  = block_size,
    d_model     = d_model,
    n_heads     = n_heads,
    n_layers    = n_layers,
    d_ff        = d_ff,
    dropout     = dropout_rate
).to(device)

## 9. Optimizer & Learning Rate Schedule
- **AdamW**: Adam + decoupled weight decay → prevents overfitting
- **Warmup**: LR ramps from 0 → peak over first N steps
- **Cosine decay**: smooth drop to 10% of peak LR

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate,
    weight_decay=weight_decay,
    betas=(0.9, 0.95)
)

def get_lr(step):
    """
    Learning rate schedule:
      - Linear warmup for `warmup_steps` steps
      - Cosine decay from peak down to 10% of peak
    """
    if step < warmup_steps:
        return learning_rate * (step + 1) / warmup_steps
    # Cosine decay
    progress = (step - warmup_steps) / max(1, max_steps - warmup_steps)
    cosine   = 0.5 * (1.0 + math.cos(math.pi * progress))
    # Decay to 10% of peak
    return learning_rate * (0.1 + 0.9 * cosine)

print('Optimizer and LR scheduler ready.')

## 10. Loss Estimation Utility
Averages loss over multiple batches for a stable estimate.

In [ ]:
@torch.no_grad()
def estimate_loss():
    """
    Estimate mean loss over `eval_iters` batches for both train and val splits.
    Model is set to eval mode (disables dropout) during this call.
    """
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = []
        for _ in range(eval_iters):
            x, y = get_batch(split)
            _, loss = model(x, y)
            losses.append(loss.item())
        out[split] = sum(losses) / len(losses)
    model.train()
    return out

## 11. Training Loop
Tracks train/val loss, applies gradient clipping, LR scheduling, and checkpointing.

In [ ]:
train_losses = []
val_losses   = []
loss_steps   = []
best_val_loss = float('inf')

print(f'Starting training for {max_steps} steps...\n')
print(f'{"Step":>6}  {"Train Loss":>10}  {"Val Loss":>10}  {"Perplexity":>10}  {"LR":>10}')
print('-' * 60)

model.train()

for step in range(max_steps):

    # ── Update learning rate ──────────────────────────────────
    lr = get_lr(step)
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr

    # ── Evaluate periodically ────────────────────────────────
    if step % eval_interval == 0 or step == max_steps - 1:
        losses = estimate_loss()
        train_losses.append(losses['train'])
        val_losses.append(losses['val'])
        loss_steps.append(step)

        ppl = math.exp(losses['val'])
        print(f'{step:>6}  {losses["train"]:>10.4f}  {losses["val"]:>10.4f}  {ppl:>10.2f}  {lr:>10.2e}')

        # ── Checkpoint: save best model ───────────────────────
        if losses['val'] < best_val_loss:
            best_val_loss = losses['val']
            torch.save(model.state_dict(), 'best_model.pt')

    # ── Forward pass ─────────────────────────────────────────
    x, y    = get_batch('train')
    _, loss = model(x, y)

    # ── Backward pass ────────────────────────────────────────
    optimizer.zero_grad(set_to_none=True)
    loss.backward()

    # ── Gradient clipping (prevents exploding gradients) ─────
    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

    optimizer.step()

print('\nTraining complete!')
print(f'Best validation loss: {best_val_loss:.4f}  (Perplexity: {math.exp(best_val_loss):.2f})')

## 12. Loss Curves — Underfitting vs Overfitting Diagnosis
- **Both losses high** → underfitting (increase model size or data)
- **Train ↓, Val ↑** → overfitting (use dropout / early stop)
- **Both low, small gap** → ideal training

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(loss_steps, train_losses, label='Train Loss', color='royalblue', linewidth=2)
plt.plot(loss_steps, val_losses,   label='Val Loss',   color='tomato',    linewidth=2)
plt.xlabel('Step')
plt.ylabel('Cross-Entropy Loss')
plt.title('GPT Training — Loss Curves')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('loss_curves.png', dpi=150)
plt.show()
print('Loss curves saved to loss_curves.png')

## 13. Text Generation
Load the best checkpoint and generate Shakespeare-style text autoregressively.

In [ ]:
# Load best checkpoint
model.load_state_dict(torch.load('best_model.pt', map_location=device))
model.eval()

def generate_text(prompt='\n', max_new_tokens=500, temperature=0.8, top_k=40):
    """
    Generate text from an optional prompt.
    Uses temperature scaling and top-k sampling.
    """
    context = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
    output  = model.generate(context, max_new_tokens=max_new_tokens,
                             temperature=temperature, top_k=top_k)
    return decode(output[0].tolist())

print('=' * 60)
print('GENERATED TEXT (temperature=0.8, top_k=40):')
print('=' * 60)
print(generate_text(prompt='\n', max_new_tokens=500))

In [ ]:
# Try with a custom prompt
print('=' * 60)
print('GENERATED TEXT — Custom Prompt:')
print('=' * 60)
print(generate_text(prompt='To be or not to be', max_new_tokens=300, temperature=0.7))

## 14. Architecture Summary

In [ ]:
print('=' * 55)
print('           GPT ARCHITECTURE SUMMARY')
print('=' * 55)
print(f'  Tokenization     : Character-level (vocab={vocab_size})')
print(f'  Block size       : {block_size} characters')
print(f'  d_model          : {d_model}')
print(f'  Attention heads  : {n_heads} (d_k = {d_model // n_heads} per head)')
print(f'  Decoder layers   : {n_layers}')
print(f'  FFN hidden dim   : {d_ff}')
print(f'  Dropout          : {dropout_rate}')
print(f'  Optimizer        : AdamW (wd={weight_decay})')
print(f'  LR schedule      : Warmup ({warmup_steps} steps) + Cosine decay')
print(f'  Grad clipping    : {grad_clip}')
total = sum(p.numel() for p in model.parameters())
print(f'  Total parameters : {total:,}')
print(f'  Best val loss    : {best_val_loss:.4f}')
print(f'  Best perplexity  : {math.exp(best_val_loss):.2f}')
print('=' * 55)